# 05 - Rule-based ensemble of IndoBERTweet, CatBoost and similarity predictions into the final submission

# Import and Install Library

# Loading Prediction

In [ ]:
DATA_DIR = "../data"

In [1]:
import pandas as pd

In [2]:
# read submission\submission_indobert_edu.csv and submission\Baseline.csv
pred_indobert = pd.read_csv(f'{DATA_DIR}/submission_indobert_edu.csv')
pred_baseline = pd.read_csv(f'{DATA_DIR}/submission_baseline.csv')
similarity = pd.read_csv(f'{DATA_DIR}/similarity_data.csv')
pred_catboost = pd.read_csv(f'{DATA_DIR}/pred_catboost.csv')
pred_ijelid = pd.read_csv(f'{DATA_DIR}/submission_ijelid.csv')

In [3]:
final_submission = pred_indobert.copy() # Final Submission berbasis pada indobert

# Ensemble

## Use Geografi from Submission Baseline

In [ ]:
pred_baseline

In [8]:
pred_baseline[pred_baseline['Kelas'] == 'Geografi']

      IDText     Kelas
23   TXT0024  Geografi
216  TXT0217  Geografi
704  TXT0705  Geografi
716  TXT0717  Geografi
728  TXT0729  Geografi
735  TXT0736  Geografi

In [9]:
# get index from baseline where the prediction is Geografi
geografi_index = pred_baseline[pred_baseline['Kelas'] == 'Geografi'].index

In [10]:
final_submission.iloc[geografi_index]

      IDText                                               text  \
23   TXT0024  Apalagi kita sebgai negara kepulauan Bagian pa...   
216  TXT0217  katanya Anies gapunya etika ke prabowo yang le...   
704  TXT0705  sia sia dong bgn ibu kota negara kl pusat ekon...   
716  TXT0717  Gue berdoa dan berharap klo nnti pa dan mas te...   
728  TXT0729  Reklamasi Jokuwi Ahok itu 17 Pulau dihentikan ...   
735  TXT0736  Kalau memang masuk ke Banten ngapain Anies sam...   

                Kelas  
23   Sumber Daya Alam  
216     Sosial Budaya  
704          Geografi  
716  Sumber Daya Alam  
728  Sumber Daya Alam  
735  Sumber Daya Alam  

In [11]:
# for all geografi_index, replace the prediction with submission prediction

for i in geografi_index:
    final_submission['Kelas'][i] = pred_baseline['Kelas'][i]

## Using Similarity for some cases (Mostly leakage)

In [12]:
# loop trough test_df_embed and submission
for i, row in similarity.iterrows():
    # if similarity.loc[i, 'train_label'] == 'Politik' and similarity.loc[i, 'cosine_sim'] >= 0.98:
    #     final_submission.loc[i,'Kelas'] = 'Politik'
    if similarity.loc[i, 'train_label'] == 'Ekonomi' and similarity.loc[i, 'cosine_sim'] >= 0.95:
        final_submission.loc[i,'Kelas'] = 'Ekonomi'
    elif similarity.loc[i, 'train_label'] == 'Geografi' and similarity.loc[i, 'cosine_sim'] >= 0.909:
        final_submission.loc[i,'Kelas'] = 'Geografi'
    elif similarity.loc[i, 'train_label'] == 'Ideologi' and similarity.loc[i, 'cosine_sim'] >= 0.938:
        final_submission.loc[i,'Kelas'] = 'Ideologi'
    elif similarity.loc[i, 'train_label'] == 'Pertahanan dan Keamanan' and similarity.loc[i, 'cosine_sim'] >= 0.952:
        final_submission.loc[i,'Kelas'] = 'Pertahanan dan Keamanan'
    elif similarity.loc[i, 'train_label'] == 'Sosial Budaya' and similarity.loc[i, 'cosine_sim'] >= 0.942:
        final_submission.loc[i,'Kelas'] = 'Sosial Budaya'

## Ekonomi, Sosial Budaya, Ideologi from another prediction

In [13]:
for i, row in pred_catboost.iterrows():
    if pred_catboost.loc[i, 'label_polos'] == 'Ekonomi' and pred_ijelid.loc[i, 'Kelas'] == 'Ekonomi' and pred_baseline.loc[i, 'Kelas'] == 'Ekonomi':
        #change value in final_submission
        if final_submission.loc[i, 'Kelas'] != 'Demografi':
            final_submission.loc[i, 'Kelas'] = 'Ekonomi'
    if pred_catboost.loc[i, 'label_polos'] == 'Sosial Budaya' and pred_ijelid.loc[i, 'Kelas'] == 'Sosial Budaya' and pred_baseline.loc[i, 'Kelas'] == 'Sosial Budaya':
        #change value in final_submission
        if final_submission.loc[i, 'Kelas'] != 'Demografi':
            final_submission.loc[i, 'Kelas'] = 'Sosial Budaya'
    if pred_catboost.loc[i, 'label_polos'] == 'Ideologi' and pred_ijelid.loc[i, 'Kelas'] == 'Ideologi' and pred_baseline.loc[i, 'Kelas'] == 'Ideologi':
        #change value in final_submission but check if it is other than Demografi
        if final_submission.loc[i, 'Kelas'] != 'Demografi':
            final_submission.loc[i, 'Kelas'] = 'Ideologi'

## Politic From Similarity but keep the small class

In [14]:
# loop trough test_df_embed and submissionv2
for i, row in similarity.iterrows():
    # print submissionv2 at index i
    if similarity.loc[i, 'train_label'] == 'Politik' and similarity.loc[i, 'cosine_sim'] >= 0.98:
        if final_submission.loc[i,'Kelas'] != 'Demografi' and final_submission.loc[i,'Kelas'] != 'Geografi' and final_submission.loc[i,'Kelas'] != 'Sumber Daya Alam':
            final_submission.loc[i,'Kelas'] = 'Politik'

# Rule Based

In [15]:
sapi_idx = final_submission[final_submission['text'].str.contains('impor sapi')].index

In [16]:
final_submission.loc[sapi_idx, 'Kelas'] = 'Sumber Daya Alam'

In [17]:
bloomberg_idx = final_submission[final_submission['text'].str.contains('bloomberg')].index

In [18]:
final_submission.loc[bloomberg_idx, 'Kelas'] = 'Politik'

# Finishing

In [19]:
final_submission.head()

    IDText                                               text  \
0  TXT0001  Lu mau orang 2 pro demokrasi di negara ini bis...   
1  TXT0002  Prabowo ditanya soal hutang luar negeri dia me...   
2  TXT0003  kiki daliyo Ganjar Pranowo itulah beliau sosok...   
3  TXT0004  Prabowo Gibran yang bisa melakukan itu semua d...   
4  TXT0005  Lah justru yang gak nyambung junjungan elu aom...   

                     Kelas  
0                  Politik  
1  Pertahanan dan Keamanan  
2                 Ideologi  
3                  Politik  
4            Sosial Budaya  

In [ ]:
final_submission.to_csv(f'{DATA_DIR}/submission_final.csv', index=False)